In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("all_vlm_models.csv",index_col='Unnamed: 0')


In [ ]:
import re

def parse_model_name(model_name):
   
    lr_match = re.search(r'lr([\d\.]+e[+-]?\d+)', model_name)
    lr = lr_match.group(1) if lr_match else 'unknown'
    
    # Extract base model (everything after seed<number>-)
    base_match = re.search(r'seed\d+-(.+\.ckpt)', model_name)
    base_model = base_match.group(1) if base_match else 'unknown'
    
    return lr, base_model

In [ ]:
models = [model for model in df.index.tolist() if model != "original" and not model.endswith("slm-1-phase-epoch9-val_loss2.435871-batch16-v1942-seed1561312.ckpt")]
model_info = []

for model in models:
    lr, base_model = parse_model_name(model)
    model_info.append({
        'model_name': model,
        'learning_rate': lr,
        'base_model': base_model,
        'lr_numeric': float(lr) if lr != 'unknown' else 0.0
    })

df_models = pd.DataFrame(model_info)

# Sort by learning rate first, then base model
df_models = df_models.sort_values(['lr_numeric', 'base_model'])



In [ ]:
df_models["model_without_base"] = df_models.apply(lambda row: row["model_name"].replace(row["base_model"], ""), axis=1)

In [ ]:
# Add LoRA rank column based on model name
df_models['lora_rank'] = df_models['model_without_base'].apply(
    lambda name: 32 if ('-32-' in name or '-r32-' in name) else 0
)


In [ ]:
# Add augmentation column based on model name
df_models['aug'] = df_models['model_without_base'].apply(
    lambda name: 0 if 'no-aug' in name else 1
)


In [ ]:
# Add base_model_freeze column based on base_model name
df_models['base_model_llm_freeze'] = df_models['base_model'].apply(
    lambda name: 1 if 'freese' in name else 0
)


In [ ]:
# Add second_phase_projection_freeze column based on model name
df_models['second_phase_projection_freeze'] = df_models['model_without_base'].apply(
    lambda name: 1 if ('proj-frozen' in name or 'freeze-llm' in name) else 0
)


In [ ]:
df_models = df_models.sort_values(by = ["aug", "lora_rank", "lr_numeric", "base_model_llm_freeze", "base_model", "second_phase_projection_freeze"], ascending = [False, False, True, True, True, False])

In [ ]:
df_models.to_csv("all_vlm_models_sorted.csv", index=False)

In [ ]:
df_models

In [ ]:
leiomioma_cols = [col for col in df.columns if col.startswith("/run/media/victor/pessoal/mestrado/dataset/leiomioma")]
df = df[leiomioma_cols]

In [ ]:
from rouge_score import rouge_scorer


rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
print("ROUGE evaluator initialized")


In [ ]:
def clean_text(text):

    if pd.isna(text):
        return ""
    text = str(text).lower()

    return text

print("Cleaning text: converting to lowercase, removing punctuation, and removing stop words...")
df = df.map(clean_text)
print("Text cleaning complete")


In [ ]:
df.isna().sum().sum()

In [ ]:
from tqdm import tqdm
import numpy as np

print("Calculating ROUGE scores for all models...")

ground_truth_row_name = 'original'

if ground_truth_row_name not in df.index:
    print(f"Available row names: {df.index.tolist()}")
else:
    ground_truth = df.loc[ground_truth_row_name]
    print(len(ground_truth))

    model_rows = [idx for idx in df.index if idx != ground_truth_row_name]
    
    rouge_scores_dict = {}
    
    for model_name in tqdm(model_rows, desc="Processing models"):
        model_predictions = df.loc[model_name]
        
        rouge1_precision_scores = []
        rouge1_recall_scores = []
        rouge1_f1_scores = []
        rouge2_precision_scores = []
        rouge2_recall_scores = []
        rouge2_f1_scores = []
        rougeL_precision_scores = []
        rougeL_recall_scores = []
        rougeL_f1_scores = []
        
        for img_path in tqdm(df.columns, desc=f"ROUGE scores for {model_name}", leave=False):
            ref_text = str(ground_truth[img_path])
            cand_text = str(model_predictions[img_path])
            
            ref_text = ref_text.replace('<|endoftext|>', '').strip()
            cand_text = cand_text.replace('<|endoftext|>', '').strip()
            
            if pd.isna(ref_text) or pd.isna(cand_text) or ref_text.strip() == '' or cand_text.strip() == '':
                print(f"  Empty candidate for model '{model_name}' on image: {img_path}")
                rouge1_precision_scores.append(np.nan)
                rouge1_recall_scores.append(np.nan)
                rouge1_f1_scores.append(np.nan)
                rouge2_precision_scores.append(np.nan)
                rouge2_recall_scores.append(np.nan)
                rouge2_f1_scores.append(np.nan)
                rougeL_precision_scores.append(np.nan)
                rougeL_recall_scores.append(np.nan)
                rougeL_f1_scores.append(np.nan)
                continue
            
            
            try:
                scores = rouge_evaluator.score(ref_text, cand_text)
                
                rouge1_precision_scores.append(scores['rouge1'].precision)
                rouge1_recall_scores.append(scores['rouge1'].recall)
                rouge1_f1_scores.append(scores['rouge1'].fmeasure)
                
                rouge2_precision_scores.append(scores['rouge2'].precision)
                rouge2_recall_scores.append(scores['rouge2'].recall)
                rouge2_f1_scores.append(scores['rouge2'].fmeasure)
                
                rougeL_precision_scores.append(scores['rougeL'].precision)
                rougeL_recall_scores.append(scores['rougeL'].recall)
                rougeL_f1_scores.append(scores['rougeL'].fmeasure)
                
            except Exception as e:
                print(f"  Error processing model '{model_name}' on image {img_path}: {e}")
                rouge1_precision_scores.append(np.nan)
                rouge1_recall_scores.append(np.nan)
                rouge1_f1_scores.append(np.nan)
                rouge2_precision_scores.append(np.nan)
                rouge2_recall_scores.append(np.nan)
                rouge2_f1_scores.append(np.nan)
                rougeL_precision_scores.append(np.nan)
                rougeL_recall_scores.append(np.nan)
                rougeL_f1_scores.append(np.nan)
                continue
        
        rouge_scores_dict[f'{model_name}_rouge1_precision'] = rouge1_precision_scores
        rouge_scores_dict[f'{model_name}_rouge1_recall'] = rouge1_recall_scores
        rouge_scores_dict[f'{model_name}_rouge1_f1'] = rouge1_f1_scores
        rouge_scores_dict[f'{model_name}_rouge2_precision'] = rouge2_precision_scores
        rouge_scores_dict[f'{model_name}_rouge2_recall'] = rouge2_recall_scores
        rouge_scores_dict[f'{model_name}_rouge2_f1'] = rouge2_f1_scores
        rouge_scores_dict[f'{model_name}_rougeL_precision'] = rougeL_precision_scores
        rouge_scores_dict[f'{model_name}_rougeL_recall'] = rougeL_recall_scores
        rouge_scores_dict[f'{model_name}_rougeL_f1'] = rougeL_f1_scores
    
    rouge_scores_df = pd.DataFrame(rouge_scores_dict, index=df.columns)
    rouge_scores_df.index.name = 'img_path'
    
    rouge_scores_df.head()


In [ ]:
model_rougeL_f1_means = []

for model_name in model_rows:
    rouge1_f1_col = f'{model_name}_rouge1_f1'
    rouge2_f1_col = f'{model_name}_rouge2_f1'
    rougeL_f1_col = f'{model_name}_rougeL_f1'
    
    mean_rouge1_f1 = rouge_scores_df[rouge1_f1_col].mean()
    mean_rouge2_f1 = rouge_scores_df[rouge2_f1_col].mean()
    mean_rougeL_f1 = rouge_scores_df[rougeL_f1_col].mean()
    model_rougeL_f1_means.append((model_name, mean_rougeL_f1))
    

print("\n" + "="*80)
print("Model Ranking by Mean ROUGE-L F1 Score:")
print("="*80)


model_rougeL_f1_means.sort(key=lambda x: x[1], reverse=True)

for rank, (model_name, mean_rougeL_f1) in enumerate(model_rougeL_f1_means, 1):
    print(f"{rank:2d}. {model_name:60s} ROUGE-L={mean_rougeL_f1:.4f}")

# Create a DataFrame with model names and their ROUGE-L F1 scores
rouge_ranking_df = pd.DataFrame(model_rougeL_f1_means, columns=['model_name', 'rougeL_f1_score'])
rouge_ranking_df = rouge_ranking_df.sort_values('rougeL_f1_score', ascending=False).reset_index(drop=True)
rouge_ranking_df['rouge_l_rank'] = range(1, len(rouge_ranking_df) + 1)
rouge_ranking_df = rouge_ranking_df[['rouge_l_rank', 'model_name', 'rougeL_f1_score']]

print("\nROUGE-L F1 Ranking DataFrame:")
print(rouge_ranking_df)


In [ ]:
rouge_ranking_df

In [ ]:
from tqdm import tqdm
import numpy as np
from bert_score import score, BERTScorer
import torch

print("Calculating BERT scores for all images...")

ground_truth_row_name = 'original'

if ground_truth_row_name not in df.index:
    print(f"Available row names: {df.index.tolist()}")
    print(f"\nPlease set ground_truth_row_name to the correct row name")
else:
    ground_truth = df.loc[ground_truth_row_name]
    
    model_rows = [idx for idx in df.index if idx != ground_truth_row_name]
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    scorer = BERTScorer(
        model_type='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
        num_layers=12,
        lang="en",
        rescale_with_baseline=False,
        device=device,
    )
    
    bert_scores_dict = {}
    
    for model_name in tqdm(model_rows, desc="Processing models"):
        model_predictions = df.loc[model_name]
        
        precision_scores = []
        recall_scores = []
        f1_scores = []
        
        for img_path in tqdm(df.columns, desc=f"BERT scores for {model_name}", leave=False):
            ref_text = str(ground_truth[img_path])
            cand_text = str(model_predictions[img_path])
            
            ref_text = ref_text.replace('<|endoftext|>', '').strip()
            cand_text = cand_text.replace('<|endoftext|>', '').strip()
            
            if pd.isna(ref_text) or pd.isna(cand_text) or ref_text.strip() == '' or cand_text.strip() == '':
                print(f"  Empty candidate for model '{model_name}' on image: {img_path}")
                precision_scores.append(np.nan)
                recall_scores.append(np.nan)
                f1_scores.append(np.nan)
                continue
            
            try:
                P, R, F1 = scorer.score([cand_text], [ref_text])
                precision_scores.append(P.item())
                recall_scores.append(R.item())
                f1_scores.append(F1.item())
            except Exception as e:
                print(f"  Error processing model '{model_name}' on image {img_path}: {e}")
                precision_scores.append(np.nan)
                recall_scores.append(np.nan)
                f1_scores.append(np.nan)
                continue
        
        bert_scores_dict[f'{model_name}_bert_precision'] = precision_scores
        bert_scores_dict[f'{model_name}_bert_recall'] = recall_scores
        bert_scores_dict[f'{model_name}_bert_f1'] = f1_scores
    
    bert_scores_df = pd.DataFrame(bert_scores_dict, index=df.columns)
    bert_scores_df.index.name = 'img_path'
    
    print(f"\n✓ Completed! Calculated BERT scores for {len(df.columns)} images across {len(model_rows)} models")
    print(f"\nBERT scores dataframe shape: {bert_scores_df.shape}")
    bert_scores_df.head()


In [ ]:
# Display summary statistics of BERT scores for each model
print("BERT Score Statistics by Model:")
print("="*80)

# Collect mean F1 scores for ranking
model_f1_means = []

for model_name in model_rows:
    f1_col = f'{model_name}_bert_f1'
    precision_col = f'{model_name}_bert_precision'
    recall_col = f'{model_name}_bert_recall'
    
    mean_f1 = bert_scores_df[f1_col].mean()
    model_f1_means.append((model_name, mean_f1))

# Display ranking of models by mean BERT F1 score
print("\n" + "="*80)
print("Model Ranking by Mean BERT F1 Score:")
print("="*80)

# Sort models by mean F1 score in descending order
model_f1_means.sort(key=lambda x: x[1], reverse=True)

# Create dataframe with rank, model name, and score
bert_ranking_data = []
for rank, (model_name, mean_f1) in enumerate(model_f1_means, 1):
    print(f"{rank:2d}. {model_name:60s} F1={mean_f1:.4f}")
    bert_ranking_data.append({
        'bert_rank': rank,
        'model_name': model_name,
        'bert_f1_score': mean_f1
    })

bert_ranking_df = pd.DataFrame(bert_ranking_data)
print("\n" + "="*80)
print("BERT Ranking DataFrame:")
print(bert_ranking_df)


In [ ]:
bert_ranking_df

In [ ]:
rouge_ranking_df

In [ ]:
df_models = df_models.merge(bert_ranking_df, on = "model_name", how = "left")
df_models = df_models.merge(rouge_ranking_df, on = "model_name", how = "left")

In [ ]:
df_models

In [ ]:
df_models[df_models["model_name"] == "slm-2-phase-lora-32-epoch2-val_loss2.856-batch18-lr1.00e-03-warmup-0-v2276-seed1561312-slm-1-phase-slm-freese-epoch6-val_loss3.098340-batch26-v2010-seed1561312.ckpt"]

In [ ]:
df_models["base_model"].drop_duplicates().to_list()

In [ ]:
df_models.loc[df_models["base_model"] == "slm-1-phase-epoch4-val_loss2.221948-batch16-v1971-seed1561312.ckpt", "exp_1_fase"] = 1
df_models.loc[df_models["base_model"] == "slm-1-phase-epoch3-val_loss2.203662-batch16-v1972-seed1561312.ckpt", "exp_1_fase"] = 2
df_models.loc[df_models["base_model"] == "slm-1-phase-slm-freese-epoch6-val_loss3.098340-batch26-v2010-seed1561312.ckpt", "exp_1_fase"] = 3
df_models.loc[df_models["base_model"] == "slm-1-phase-slm-freese-epoch5-val_loss3.067149-batch26-v2011-seed1561312.ckpt", "exp_1_fase"] = 4


In [ ]:
df_models = df_models.sort_values(by = ["aug", "lora_rank", "lr_numeric", "base_model_llm_freeze", "exp_1_fase", "second_phase_projection_freeze", ], ascending = [False, False, True, True, True, False])

In [ ]:
df_models["exp_2_fase"] = range(1, len(df_models) + 1)


In [ ]:
df_models.sort_values(by = ["rouge_l_rank"], ascending = True)[["exp_2_fase","model_name", "rouge_l_rank"]].head(6).to_csv("teste.csv", index=False)

In [ ]:
fase_2_exps_dF = df_models[["rouge_l_rank", "rougeL_f1_score", "exp_2_fase", "learning_rate", "lora_rank", "aug", "second_phase_projection_freeze", "exp_1_fase"]].copy()

In [ ]:
fase_2_exps_dF["exp_1_fase"] = fase_2_exps_dF["exp_1_fase"].astype(int)

In [ ]:
fase_2_exps_dF["second_phase_projection_freeze"] = fase_2_exps_dF["second_phase_projection_freeze"].map({1: "Sim", 0: "Não"})
fase_2_exps_dF

In [ ]:
fase_2_exps_dF["lora_rank"] = fase_2_exps_dF["lora_rank"].map({32: "Sim", 0: "Não"})


In [ ]:
fase_2_exps_dF


In [ ]:
fase_2_exps_dF["aug"] = fase_2_exps_dF["aug"].map({1: "Sim", 0: "Não"})


In [ ]:
fase_2_exps_dF